# Derivatives Pricing Engine
Reproducible experiments; pricing logic stays in the package. Run from this folder or the project root after `pip install -e ".[dev,analysis]"`. Fixed seeds; no network access during execution.

In [ ]:
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from derivative_pricer import (
    black_scholes_price,
    analytical_greeks,
    finite_difference_greeks,
    implied_volatility,
    monte_carlo_price,
    simulate_delta_hedging,
)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
BASE = dict(spot=100, strike=100, maturity=1, rate=0.05, volatility=0.20)
NAMES = ("delta", "gamma", "vega", "theta", "rho")
plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)


def save(fig, name):
    fig.savefig(FIGURES / name, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


print(f"Call: {black_scholes_price(**BASE):.6f}")
print(f"Put: {black_scholes_price(**BASE, option_type='put'):.6f}")


## Greeks versus spot
Raw derivatives: Vega/Rho per 1.0 change, Theta per year elapsed. Multiply Vega/Rho by 0.01 for one percentage point.

In [ ]:
spots = np.linspace(60, 140, 161)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), layout="constrained")
for kind in ("call", "put"):
    results = []
    for spot in spots:
        inputs = BASE.copy()
        inputs["spot"] = spot
        results.append(analytical_greeks(**inputs, option_type=kind))
    for ax, name in zip(axes.flat, NAMES):
        ax.plot(spots, [getattr(g, name) for g in results], label=kind)
        ax.set(title=name.capitalize(), xlabel="Spot", ylabel="Raw sensitivity")
        ax.grid(alpha=0.2)
        ax.legend()
axes.flat[-1].axis("off")
fig.suptitle("Black-Scholes sensitivities | K=100, T=1, r=5%, q=0, volatility=20%")
save(fig, "greeks_vs_spot.png")


## Finite-difference error
Absolute bumps from 1e-1 to 1e-8, in the corresponding parameter units. Smaller is not always better; Gamma amplifies roundoff through division by h².

In [ ]:
bumps = np.logspace(-1, -8, 8)
exact = analytical_greeks(**BASE)
rows = []
for h in bumps:
    approx = finite_difference_greeks(
        **BASE, spot_bump=h, volatility_bump=h, maturity_bump=h, rate_bump=h
    )
    row = {"bump": h}
    for name in NAMES:
        estimated_value = getattr(approx, name)
        analytical_value = getattr(exact, name)
        row[name] = abs(estimated_value - analytical_value)
    rows.append(row)
errors = pd.DataFrame(rows).set_index("bump")
display(errors)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), layout="constrained")
units = {
    "delta": "Spot (currency)",
    "gamma": "Spot (currency)",
    "vega": "Volatility (decimal)",
    "theta": "Maturity (years)",
    "rho": "Rate (decimal)",
}
for ax, name in zip(axes.flat, NAMES):
    ax.loglog(bumps, errors[name].replace(0, np.nan), "o-")
    ax.set(
        title=name.capitalize(),
        xlabel=f"Absolute bump: {units[name]}",
        ylabel="Absolute error",
    )
    ax.grid(alpha=0.2)
axes.flat[-1].axis("off")
fig.suptitle("Central differences | independent vertical scales; zero errors omitted")
save(fig, "finite_difference_errors.png")


## Implied-volatility recovery
Synthetic prices generated at 27% volatility. Newton uses analytical Vega; out-of-bracket steps and tiny Vega trigger bisection. Very low Vega can make IV poorly identified even when the price is accurate.

In [ ]:
rows = []
for kind in ("call", "put"):
    for strike, maturity in [(80, 1), (100, 1), (120, 1), (100, 0.01), (120, 5)]:
        inputs = dict(
            spot=100,
            strike=strike,
            maturity=maturity,
            rate=0.03,
            option_type=kind,
            dividend_yield=0.01,
        )
        market = black_scholes_price(**inputs, volatility=0.27)
        recovered = implied_volatility(
            market_price=market, **inputs, initial_volatility=0.001
        )
        rows.append(
            dict(
                option=kind,
                strike=strike,
                maturity=maturity,
                price=market,
                recovered_iv=recovered.volatility,
                iterations=recovered.iterations,
                bisections=recovered.bisection_steps,
                error=abs(recovered.volatility - 0.27),
            )
        )
iv_table = pd.DataFrame(rows)
display(iv_table)
assert iv_table.error.max() < 1e-8
fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
ax.plot(
    np.arange(len(iv_table)), 100 * iv_table.recovered_iv, "o", label="Recovered IV"
)
ax.axhline(27, color="black", linestyle="--", label="Input IV = 27%")
ax.set_xticks(
    np.arange(len(iv_table)),
    [f"{r.option}\nK={r.strike}, T={r.maturity:g}" for r in iv_table.itertuples()],
    fontsize=8,
)
ax.set(
    ylabel="Volatility (%)",
    title="Implied-volatility recovery from a poor initial guess",
)
ax.set_ylim(26.5, 27.5)
ax.ticklabel_format(axis="y", style="plain", useOffset=False)
ax.legend()
save(fig, "implied_vol_recovery.png")


## Monte Carlo convergence
Exact terminal GBM draws: no time-discretization error. The standard error scales as N⁻¹ᐟ²; the realized absolute error need not decrease monotonically. Confidence intervals use an asymptotic normal approximation.

In [ ]:
counts = [100, 1_000, 10_000, 100_000, 1_000_000]
benchmark = black_scholes_price(**BASE)
rows = []
for count in counts:
    result = monte_carlo_price(**BASE, n_paths=count, seed=42)
    rows.append(
        dict(
            paths=count,
            price=result.price,
            standard_error=result.standard_error,
            lower=result.confidence_interval[0],
            upper=result.confidence_interval[1],
            absolute_error=abs(result.price - benchmark),
        )
    )
mc_table = pd.DataFrame(rows)
display(mc_table)
tail = mc_table.tail(3)
slope = np.polyfit(np.log(tail.paths), np.log(tail.standard_error), 1)[0]
print(f"Log-log standard-error slope (N >= 10,000): {slope:.3f} (theoretical -0.5)")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].errorbar(
    mc_table.paths,
    mc_table.price,
    yerr=mc_table.upper - mc_table.price,
    fmt="o",
    capsize=4,
    label="MC, 95% CI",
)
axes[0].axhline(benchmark, color="black", linestyle="--", label="Black-Scholes")
axes[0].set(
    xscale="log",
    xlabel="Paths",
    ylabel="Option price",
    title="Price and sampling uncertainty",
)
axes[0].legend()
axes[1].loglog(
    mc_table.paths, mc_table.standard_error, "o-", label="Estimated standard error"
)
axes[1].loglog(
    mc_table.paths, mc_table.absolute_error, "s--", label="Realized absolute error"
)
axes[1].loglog(
    counts,
    mc_table.standard_error.iloc[0] * np.sqrt(counts[0] / np.array(counts)),
    ":",
    label="N^-1/2 reference",
)
axes[1].set(xlabel="Paths", ylabel="Price error", title="Sampling convergence")
axes[1].legend()
save(fig, "monte_carlo_convergence.png")


## Delta hedging: short one call
Premium and Delta use 20% implied volatility. Realized volatility is 15%, 20% or 25%. Cash earns r; continuous dividends are reinvested in stock. No transaction costs. P&L is measured at expiry, per option unit. Daily/weekly/monthly mean every 1/5/21 steps of a 252-step year, with final liquidation at expiry. A shared seed couples the underlying paths across hedge frequencies. Default drift is r−q, not an estimated physical return.

In [ ]:
hedges = {}
rows = []
for realized in (0.15, 0.20, 0.25):
    for frequency, every in [("Daily", 1), ("Weekly", 5), ("Monthly", 21)]:
        result = simulate_delta_hedging(
            spot=100,
            strike=100,
            maturity=1,
            rate=0.03,
            implied_volatility=0.20,
            realized_volatility=realized,
            n_paths=10_000,
            n_steps=252,
            rebalance_every=every,
            seed=123,
        )
        hedges[(realized, frequency)] = result.pnl
        pnl_mean = result.pnl.mean()
        pnl_standard_deviation = result.pnl.std(ddof=1)
        mean_standard_error = pnl_standard_deviation / np.sqrt(result.pnl.size)
        rows.append(
            dict(
                realized_vol=realized,
                frequency=frequency,
                mean=pnl_mean,
                std=pnl_standard_deviation,
                mean_se=mean_standard_error,
                p05=np.quantile(result.pnl, 0.05),
                p95=np.quantile(result.pnl, 0.95),
            )
        )
hedge_table = pd.DataFrame(rows)
display(hedge_table)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for frequency in ("Daily", "Weekly", "Monthly"):
    subset = hedge_table[hedge_table.frequency == frequency]
    axes[0].errorbar(
        100 * subset.realized_vol,
        subset["mean"],
        yerr=1.96 * subset.mean_se,
        marker="o",
        capsize=3,
        label=frequency,
    )
    axes[1].plot(100 * subset.realized_vol, subset["std"], "o-", label=frequency)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(
    xlabel="Realized volatility (%)",
    ylabel="Mean terminal P&L",
    title="Short call | mean and 95% CI of mean",
)
axes[1].set(
    xlabel="Realized volatility (%)",
    ylabel="P&L standard deviation",
    title="Dispersion of terminal hedging P&L",
)
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
save(fig, "hedging_pnl.png")
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
for realized in (0.15, 0.20, 0.25):
    ax.hist(
        hedges[(realized, "Daily")],
        bins=70,
        density=True,
        histtype="step",
        linewidth=1.5,
        label=f"Realized {realized:.0%}",
    )
ax.set(
    xlabel="Terminal P&L",
    ylabel="Density",
    title="Daily hedge | implied volatility 20%",
)
ax.legend()
save(fig, "hedging_distribution.png")


## Real-market comparison (optional cached snapshot)
Fetch explicitly with `python -m derivative_pricer.market_data --rate 0.03 --dividend-yield 0.012`. These are illustrative rate/yield assumptions, not live curves. The command saves a small local JSON snapshot, ignored by Git. SPXW options are European and PM-settled; maturity uses ACT/365 and a 16:00 New York expiry approximation. Quotes are delayed and may be asynchronous. Historical volatility and implied volatility measure different things; model-minus-mid is not an arbitrage signal.

In [ ]:
cache = ROOT / "data" / "market_snapshot.json"
if cache.exists():
    market = json.loads(cache.read_text())
    display(
        pd.Series({k: v for k, v in market.items() if k not in ("history", "sources")})
    )
    history = pd.DataFrame(market["history"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    axes[0].plot(pd.to_datetime(history.date), history.close)
    axes[0].set(title="SPX historical closes", ylabel="Index level", xlabel="Date")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(
        ["Bid", "Mid", "Ask", "BS at historical vol"],
        [
            market["bid"],
            market["mid"],
            market["ask"],
            market["model_price_at_historical_vol"],
        ],
    )
    axes[1].set(title=market["symbol"], ylabel="Option price (index points)")
    axes[1].tick_params(axis="x", rotation=15)
    save(fig, "market_comparison.png")
else:
    print(
        "No market snapshot: run the explicit fetch command above. Offline model analyses are complete."
    )


## Interpretation
- Analytical and numerical Greeks agree only within numerical tolerances; a tiny bump can destroy Gamma accuracy.
- Newton is fast with informative Vega; the bracket safeguards difficult starting guesses.
- Monte Carlo uncertainty shrinks as the inverse square root of sample size, not linearly.
- A more frequent hedge reduces discretization risk, but does not remove a volatility-model mismatch.
- Historical volatility is backward-looking; implied volatility is conditional on both market price and model assumptions.